In [1]:
from pymilvus import MilvusClient

milvus_client = MilvusClient("./milvus_rag.db")

In [2]:
from pymilvus import FieldSchema, DataType, CollectionSchema

VECTOR_LENGTH = 768  # check the dimensionality for Silver Retriever Base (v1.1) model

id_field = FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, description="Primary id")
text = FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096, description="Page text")
embedding_text = FieldSchema("embedding", dtype=DataType.FLOAT_VECTOR, dim=VECTOR_LENGTH, description="Embedded text")

fields = [id_field, text, embedding_text]

schema = CollectionSchema(fields=fields, auto_id=True, enable_dynamic_field=True, description="RAG Texts collection")

In [3]:
COLLECTION_NAME = "rag_texts_and_embeddings"

milvus_client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema
)

index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="embedding", 
    index_type="HNSW",
    metric_type="L2",
    params={"M": 4, "efConstruction": 64}  # lower values for speed
) 

milvus_client.create_index(
    collection_name=COLLECTION_NAME,
    index_params=index_params
)

# checkout our collection
print(milvus_client.list_collections())

# describe our collection
print(milvus_client.describe_collection(COLLECTION_NAME))

2026-11-06 23:31:21,478 [ERROR][_log_rpc_error]: RPC error: [create_collection], <MilvusException: (code=1, message=collection 'rag_texts_and_embeddings' already exists)>, <elapsed:1.8ms>
Traceback:
Traceback (most recent call last):
  File "/home/janwo/Dokumenty/MLOps/MLOps_labs_26-27/lab4/.venv/lib/python3.12/site-packages/pymilvus/decorators.py", line 518, in handler
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/janwo/Dokumenty/MLOps/MLOps_labs_26-27/lab4/.venv/lib/python3.12/site-packages/pymilvus/decorators.py", line 565, in handler
    return func(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/janwo/Dokumenty/MLOps/MLOps_labs_26-27/lab4/.venv/lib/python3.12/site-packages/pymilvus/decorators.py", line 456, in handler
    raise e from e
  File "/home/janwo/Dokumenty/MLOps/MLOps_labs_26-27/lab4/.venv/lib/python3.12/site-packages/pymilvus/decorators.py", line 419, in handler
    return func(*args, **kwargs)
           ^^^

MilvusException: <MilvusException: (code=1, message=collection 'rag_texts_and_embeddings' already exists)>

In [4]:
# define data source and destination
## the document origin destination from which document will be downloaded 
pdf_url = "https://www.iab.org.pl/wp-content/uploads/2024/04/Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the document
file_name = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the processed document 
file_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.json"

## local destination of the embedded pages of the document
embeddings_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska-Embeddings.json"

## local destination of all above local required files
data_dir = "./data"

In [5]:
# download data
import os
import requests

def download_pdf_data(pdf_url: str, file_name: str) -> None:
    response = requests.get(pdf_url, stream=True)
    os.makedirs(data_dir, exist_ok=True)
    with open(os.path.join(data_dir, file_name), "wb") as file:
        for block in response.iter_content(chunk_size=1024):
            if block:
                file.write(block)

download_pdf_data(pdf_url, file_name)

In [6]:
# prepare data

import fitz
import json


def extract_pdf_text(file_name, file_json):
    document = fitz.open(os.path.join(data_dir, file_name))
    pages = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        page_text = page.get_text()
        pages.append({"page_num": page_num, "text": page_text})

    with open(os.path.join(data_dir, file_json), "w") as file:
        json.dump(pages, file, indent=4, ensure_ascii=False)

extract_pdf_text(file_name, file_json)

/home/janwo/Dokumenty/MLOps/MLOps_labs_26-27/lab4/.venv/lib/python3.12/site-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [7]:
# vectorize data

import torch
import numpy as np
from sentence_transformers import SentenceTransformer


def generate_embeddings(file_json, embeddings_json, model):
    pages = []
    with open(os.path.join(data_dir, file_json), "r") as file:
        data = json.load(file)

    for page in data:
        pages.append(page["text"])

    embeddings = model.encode(pages)

    embeddings_paginated = []
    for page_num in range(len(embeddings)):
        embeddings_paginated.append({"page_num": page_num, "embedding": embeddings[page_num].tolist()})

    with open(os.path.join(data_dir, embeddings_json), "w") as file:
        json.dump(embeddings_paginated, file, indent=4, ensure_ascii=False)

model_name = "ipipan/silver-retriever-base-v1.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)
generate_embeddings(file_json, embeddings_json, model)

/home/janwo/Dokumenty/MLOps/MLOps_labs_26-27/lab4/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 10340.27it/s]


In [8]:
def insert_embeddings(file_json, embeddings_json, client=milvus_client):
    rows = []
    with open(os.path.join(data_dir, file_json), "r") as t_f, open(os.path.join(data_dir, embeddings_json), "r") as e_f:
        text_data, embedding_data = json.load(t_f), json.load(e_f)
        text_data =  list(map(lambda d: d["text"], text_data))
        embedding_data = list(map(lambda d: d["embedding"], embedding_data))
        
        for page, (text, embedding) in enumerate(zip(text_data, embedding_data)):
            rows.append({"text":text, "embedding": embedding})

    client.insert(collection_name="rag_texts_and_embeddings", data=rows)


insert_embeddings(file_json, embeddings_json)

# load inserted data into memory
milvus_client.load_collection("rag_texts_and_embeddings")

In [9]:
# search

def search(model, query, client=milvus_client):
    embedded_query = model.encode(query).tolist()
    result = client.search(
        collection_name="rag_texts_and_embeddings", 
        data=[embedded_query], 
        limit=1,
        search_params={"metric_type": "L2"},
        output_fields=["text"]
    )
    return result


result = search(model, query="Czym jest sztuczna inteligencja")

In [10]:
import os
from google import genai
from dotenv import load_dotenv

load_dotenv()

GEMINI_KEY = os.getenv("API_KEY")
gemini_client = genai.Client(api_key=GEMINI_KEY)

#MODEL = "gemini-2.0-flash"
MODEL = "gemini-3.5-flash"

def generate_response(prompt: str):
    try:
        # Send request to Gemini 2.0 Flash API and get the response
        response = gemini_client.models.generate_content(
            model=MODEL,
            contents=prompt,
        )
        return response.text
    except Exception as e:
        print(f"Error generating response: {e}")
        return None



In [25]:
def build_prompt(context: str, query: str) -> str:
    prompt = f"""You are a helpful AI assistant. Answer the following question based solely on the provided context.
If the context does not contain sufficient information to answer the question, state explicitly: "I could not find the answer in the provided materials."

Context:
{context}

Question: {query}
Answer:"""
    return prompt


def rag(model, query: str) -> str:
    context = search(model, query)[0][0]["entity"]["text"]
    prompt = build_prompt(context, query)
    response = generate_response(prompt)
    return response


In [26]:
rag(model, "Opisz czym jest sztuczna inteligencja w 5 zdaniach")

'Oto opis sztucznej inteligencji przygotowany na podstawie tekstu, złożony z 5 zdań:\n\n1. Sztuczna inteligencja (AI) to program lub maszyna symulująca inteligencję naturalną, ludzką.\n2. Jest to obszar informatyki skupiający się na tworzeniu programów komputerowych zdolnych do wykonywania zadań wymagających ludzkiej inteligencji.\n3. Zadania te obejmują między innymi rozpoznawanie wzorców, rozumienie języka naturalnego, podejmowanie decyzji, uczenie się oraz planowanie.\n4. Głównym celem AI jest stworzenie systemów realizujących funkcje umysłu, które są zdolne do myślenia i podejmowania decyzji na sposób przypominający ludzki.\n5. Należy jednak pamiętać, że współczesne systemy sztucznej inteligencji są inteligentne tylko w ograniczonym obszarze.'

In [27]:
rag(model, "Opisz czym jest sztuczna inteligencja w maksymalnie 200 znakach")

'Sztuczna inteligencja (AI) to obszar informatyki, który skupia się na tworzeniu programów komputerowych zdolnych do wykonywania zadań, które wymagają ludzkiej inteligencji.'

In [28]:
rag(model, "Która uczelnia techniczna w Polsce jest najlepsza?")

'I could not find the answer in the provided materials.'

In [30]:
rag(model, "Describe what AI is in Polish")

'Na podstawie dostarczonego tekstu sztuczną inteligencję (AI) można opisać w zależności od jej rodzaju:\n\n* **Sztuczna inteligencja o wąskim zakresie:** jest skonstruowana do wykonywania konkretnych, określonych zadań.\n* **Sztuczna inteligencja ogólnego przeznaczenia (AGI / silna AI):** to forma AI mająca zdolność do wykonywania dowolnego zadania intelektualnego, które człowiek jest w stanie wykonać. Posiada potencjał do zrozumienia, uczenia się, adaptacji i stosowania wiedzy w szerokim zakresie zadań oraz ewoluowania w miarę napotykania nowych wyzwań i sytuacji.'

I1106 23:45:25.251388   37706 chttp2_transport.cc:1403] ipv4:127.0.0.1:36855: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E1106 23:45:25.251516   37706 chttp2_transport.cc:1435] ipv4:127.0.0.1:36855: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 20000ms
